# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Prabhaditya003/FlyRank.ai-internship-work-week1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**My answers:**

- **Unit of analysis:** one row = one content item's performance on one day, for one client — the `(client_id, content_id, report_date)` grain of `fact_content_daily_performance`.
- **Table(s):** `fact_content_daily_performance`, month=2026-03 partition (mid-panel, not `_sample`) for features; the same table's month=2026-04 partition to build the label; `dim_content` for content metadata; `dim_clients` for per-client history flags.
- **Time window:** features come only from `report_date` in March 2026 (2026-03-01 to 2026-03-31). The label looks at April 2026 (2026-04-01 to 2026-04-30) — strictly after the feature window, so nothing from the label period can leak backward into a feature.
- **What I'd predict:** for each content item, whether its average daily organic clicks in April fall 20%+ versus its March average — a "declining" flag meant as a triage signal, not a guaranteed forecast.
- **What I deliberately exclude:** `fact_content_query_90d`. Its 90-day window overlaps the snapshot's final months, so pulling it into a March-vs-April comparison risks mixing information from inside my label window into what I'm calling a "feature."

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Setup: auth + connection. Real verification queries live in section 3.
from google.colab import userdata
import os
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

# HF_TOKEN comes from Colab Secrets — never paste it in a cell (this is a public repo).
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

import duckdb
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

con.execute(f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE huggingface,
        TOKEN '{os.environ["HF_TOKEN"]}'
    )
""")

REPO = "hf://datasets/FlyRank/internship-warehouse"
MARCH = f"{REPO}/fact_content_daily_performance/month=2026-03/*.parquet"
APRIL = f"{REPO}/fact_content_daily_performance/month=2026-04/*.parquet"
DIM_CONTENT = f"{REPO}/dim_content.parquet"
DIM_CLIENTS = f"{REPO}/dim_clients.parquet"

print("connected, partitions set")


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


connected, partitions set


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature — five, max, all knowable by end-of-March (the decision moment):**

1. `march_avg_daily_clicks` — mean of `clicks` over March rows for that content item (GSC).
2. `march_avg_gsc_position` — mean of `gsc_avg_position` over March. `0` means "no data," not rank zero — same gotcha as the starter CSV's `avg_position`.
3. `march_ctr` — March `clicks` ÷ March `impressions` (GSC).
4. `ga4_data_available` — the flag itself, used as a feature: which data regime a content item is in is knowable in real time, not after the fact.
5. `content_word_count` (from `dim_content`) — a static content attribute known long before March. Missing values get a companion `has_word_count` flag instead of a silent 0, since missingness follows `content_type`.

**Label / proxy:**
- `is_declining_april` — 1 if April's average daily clicks < 0.8 × March's average daily clicks, else 0. Built entirely from `clicks`; never a feature.

**Context (grouping/joining only, never learned from):**
- `client_id`, `content_id` — pseudonymous join/group keys.
- `report_date`, `month` partition.
- `dim_clients.gsc_data_start`, `dim_clients.ga4_data_start` — decide which clients/rows are even eligible; never fed to a model.

**Excluded (with why):**
- `fact_content_query_90d` — its 90-day window overlaps the April label period; including it risks leaking April-adjacent signal into "March" features.
- Raw GA4 columns (engagement, scroll_rate, etc.) for rows where `ga4_data_available = FALSE` — zero-filled, not real zeros.
- Anything computed the same way as my label (the starter-CSV's `trend_direction`/`trend_pct` trap, generalized) — excluded on principle even where the warehouse doesn't ship the identical column name.
- Client names / URLs — privacy; never present in what I output.

In [21]:
con.execute(f"DESCRIBE SELECT * FROM '{MARCH}'").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Build the March feature frame (five features) + April label_kept in separate steps
# so it's obvious the label never touches the feature-building query.

features_q = f"""
SELECT
    client_hash_id,
    content_hash_id,
    AVG(gsc_clicks)                      AS march_avg_daily_clicks,
    AVG(NULLIF(gsc_sum_position, 0))     AS march_avg_gsc_position,
    SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions),0) AS march_ctr,
    BOOL_OR(ga4_data_available)          AS ga4_data_available
FROM '{MARCH}'
GROUP BY client_hash_id, content_hash_id
"""

features_df = con.execute(features_q).df()

content_meta = con.execute(f"SELECT content_hash_id, word_count FROM '{DIM_CONTENT}'").df()
content_meta["has_word_count"] = content_meta["word_count"].notna()

features_df = features_df.merge(content_meta, on="content_hash_id", how="left")

label_q = f"""
SELECT client_hash_id, content_hash_id, AVG(gsc_clicks) AS april_avg_daily_clicks
FROM '{APRIL}'
GROUP BY client_hash_id, content_hash_id
"""

label_df = con.execute(label_q).df()

contract_df = features_df.merge(label_df, on=["client_hash_id", "content_hash_id"], how="inner")
contract_df["is_declining_april"] = (
    contract_df["april_avg_daily_clicks"] < 0.8 * contract_df["march_avg_daily_clicks"]
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Three checks, on the March 2026 partition (mid-panel, not `_sample`):**

1. **Grain** — group by `(report_date, client_id, content_id)` and confirm nothing has count > 1.
2. **Row count + date span** — total rows in `month=2026-03`, plus `MIN`/`MAX(report_date)`, checked against the warehouse's stated scale.
3. **Availability** — how many March rows have `ga4_data_available = TRUE` vs `FALSE`, filtered with `IS TRUE`, since that flag is one of my five features.

**The trap (on purpose):** add one label-derived column straight into the feature frame — `april_avg_daily_clicks` itself — and score how well it "predicts" `is_declining_april`. It should look almost perfect, because it's built from the same numbers as the label. Delete it, and report the honest number using only the five March-only features.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1. Grain check — expect zero rows back
# 1. Grain check – expect zero rows back
grain_check = con.execute(f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
FROM '{MARCH}'
GROUP BY report_date, client_hash_id, content_hash_id
HAVING c > 1
LIMIT 5
""").df()
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,c


In [24]:
features_q = f"""
SELECT
    client_hash_id,
    content_hash_id,
    AVG(gsc_clicks)                      AS march_avg_daily_clicks,
    AVG(NULLIF(gsc_sum_position, 0))     AS march_avg_gsc_position,
    SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions),0) AS march_ctr,
    BOOL_OR(ga4_data_available)          AS ga4_data_available
FROM '{MARCH}'
GROUP BY client_hash_id, content_hash_id
"""

features_df = con.execute(features_q).df()

content_meta = con.execute(f"SELECT content_hash_id, word_count FROM '{DIM_CONTENT}'").df()
content_meta["has_word_count"] = content_meta["word_count"].notna()

features_df = features_df.merge(content_meta, on="content_hash_id", how="left")

label_q = f"""
SELECT client_hash_id, content_hash_id, AVG(gsc_clicks) AS april_avg_daily_clicks
FROM '{APRIL}'
GROUP BY client_hash_id, content_hash_id
"""

label_df = con.execute(label_q).df()

contract_df = features_df.merge(label_df, on=["client_hash_id", "content_hash_id"], how="inner")
contract_df["is_declining_april"] = (
    contract_df["april_avg_daily_clicks"] < 0.8 * contract_df["march_avg_daily_clicks"]
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [25]:
# 3. GA4 availability — filtered with IS TRUE, not a truthy check
# 1. Grain check – expect zero rows back
grain_check = con.execute(f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
FROM '{MARCH}'
GROUP BY report_date, client_hash_id, content_hash_id
HAVING c > 1
LIMIT 5
""").df()
grain_check
counts = con.execute(f"""
SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date
FROM '{MARCH}'
""").df()
counts

# 3. GA4 availability – filtered with IS TRUE, not a truthy check
availability = con.execute(f"""
SELECT
    COUNT(*) AS n_rows,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS n_available,
    AVG(CASE WHEN ga4_data_available IS TRUE THEN 1.0 ELSE 0 END) AS pct_available
FROM '{MARCH}'
""").df()
availability
# ========== CELL 4: Model Comparison (Leaky vs Honest) ==========
# The trap: add the label-derived column on purpose, watch the score jump, then remove it.
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

honest_features = [
    "march_avg_daily_clicks", "march_avg_gsc_position", "march_ctr",
    "ga4_data_available", "has_word_count",
]
df = contract_df.dropna(subset=honest_features + ["is_declining_april"]).copy()

# --- WITH the leak ---
leaky_features = honest_features + ["april_avg_daily_clicks"]
X_train, X_test, y_train, y_test = train_test_split(
    df[leaky_features], df["is_declining_april"], test_size=0.2, random_state=0
)
leaky_model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
leaky_acc = accuracy_score(y_test, leaky_model.predict(X_test))

# --- WITHOUT the leak (the honest number) ---
X_train, X_test, y_train, y_test = train_test_split(
    df[honest_features], df["is_declining_april"], test_size=0.2, random_state=0
)
honest_model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
honest_acc = accuracy_score(y_test, honest_model.predict(X_test))

print(f"accuracy WITH leaked april_avg_daily_clicks: {leaky_acc:.3f}")
print(f"accuracy with honest March-only features:   {honest_acc:.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

accuracy WITH leaked april_avg_daily_clicks: 0.964
accuracy with honest March-only features:   0.769


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- **History is uneven per client.** `gsc_data_start` varies a lot; a client that only joined in May 2026 has no March row at all, so my March-based features silently drop them — this isn't a random sample of clients, it's whoever has March history.
- **GA4 rows before `ga4_data_start` are zero-filled, not missing-flagged, in the raw columns** — without checking `ga4_data_available IS TRUE`, "0 engagement" looks identical to "no engagement," which is why that flag is a feature rather than a filter I apply once and forget.
- **A 20%-drop label is a rough proxy, not a diagnosis.** It can't tell seasonal dips, a client pausing content work, and a real ranking loss apart — all it says is "clicks fell by this much," decision-support only.
- **March→April is one 30-day hop in a 17-month panel.** Nothing here says whether the same threshold behaves the same way for a different month pair, or whether this particular decline rate is typical or unusual across the panel.
- **Content-day grain hides within-day query mix.** `fact_content_query_90d` would show which queries drove the clicks, but I excluded it for window-overlap reasons (section 2), so this contract can say a content item declined, not why.

In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Back up the "uneven history" limit with a number: share of clients whose
# GSC history doesn't even reach the March partition.
history_check = con.execute(f"""
SELECT
    COUNT(*) AS n_clients,
    SUM(CASE WHEN gsc_data_start > DATE '2026-03-01' THEN 1 ELSE 0 END) AS n_after_march_start,
    AVG(CASE WHEN gsc_data_start > DATE '2026-03-01' THEN 1.0 ELSE 0 END) AS pct_after_march_start
FROM '{DIM_CLIENTS}'
""").df()
history_check

,n_clients,n_after_march_start,pct_after_march_start
0,104,15.0,0.144231


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.